### Add and mark an optional table-name parameter

In [ ]:
table_name = "sales_data_stage"

StatementMeta(, 6bbf353b-472d-4a4d-9179-b010229faf3a, 3, Finished, Available, Finished, False)

### Read the CSV with headers and verify the schema

In [2]:
from pyspark.sql import functions as F
source_path = "Files/source_files_sales/SalesData.csv"
df = (spark.read.format("csv")
.option("header", "true")
.option("inferSchema", "false")
.load(source_path))
print("Rows read:", df.count())
print("Columns:", df.columns)
display(df.limit(10))

StatementMeta(, 6bbf353b-472d-4a4d-9179-b010229faf3a, 4, Finished, Available, Finished, False)

Rows read: 32718
Columns: ['SalesOrderNumber', 'SalesOrderLineNumber', 'OrderDate', 'CustomerName', 'EmailAddress', 'Item', 'Quantity', 'UnitPrice', 'TaxAmount']


SynapseWidget(Synapse.DataFrame, c149d6dc-2f25-46bd-a863-5ac0f75c09ac)

### Assign correct types and add Year / Month

In [3]:
df_stage = (df
.withColumn("SalesOrderLineNumber", F.col("SalesOrderLineNumber").cast("int"))
.withColumn("OrderDate", F.to_date("OrderDate", "yyyy-MM-dd"))
.withColumn("Quantity", F.col("Quantity").cast("int"))
.withColumn("UnitPrice", F.col("UnitPrice").cast("decimal(18,4)"))
.withColumn("TaxAmount", F.col("TaxAmount").cast("decimal(18,4)"))
.withColumn("Year", F.year("OrderDate"))
.withColumn("Month", F.month("OrderDate")))
display(df_stage.limit(10))

StatementMeta(, 6bbf353b-472d-4a4d-9179-b010229faf3a, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f1d64fbd-8efe-4e8b-8b6f-07e62684dc3c)

### Validate the transformed DataFrame

In [4]:
print("Count:", df_stage.count())
df_stage.groupBy("Year").count().orderBy("Year").show()
print("Invalid order dates:", df_stage.filter(F.col("OrderDate").isNull()).count())

StatementMeta(, 6bbf353b-472d-4a4d-9179-b010229faf3a, 6, Finished, Available, Finished, False)

Count: 32718
+----+-----+
|Year|count|
+----+-----+
|2019| 1201|
|2020| 2733|
|2021|28784|
+----+-----+

Invalid order dates: 0


### Write the Delta staging table

In [5]:
(df_stage.write.format("delta")
.mode("overwrite")
.option("overwriteSchema", "true")
.saveAsTable(table_name))

StatementMeta(, 6bbf353b-472d-4a4d-9179-b010229faf3a, 7, Finished, Available, Finished, False)